# Calibration Pipeline Internals

Internal reference for debugging and development of the calibration pipeline.

**Requirements:** `policy_data.db`, `block_cd_distributions.csv.gz`, and the stratified CPS H5 file in `STORAGE_FOLDER`.

---
# Part 1: The Calibration Matrix

The calibration pipeline has three stages: (1) compute uprated target values, (2) assemble the sparse constraint matrix, and (3) optimize weights (`unified_calibration.py`). This section is the diagnostic checkpoint between stages 1 and 2 — understand your matrix before you optimize.

We build the full calibration matrix using `UnifiedMatrixBuilder` with clone-based geography from `assign_random_geography`, then inspect its structure: what rows and columns represent, how target groups partition the loss function, and where sparsity patterns emerge.

**Column layout:** `col = clone_idx * n_records + record_idx`

## 1.1 Setup

In [1]:
import numpy as np
import pandas as pd
from policyengine_us import Microsimulation
from policyengine_us_data.storage import STORAGE_FOLDER
from policyengine_us_data.calibration.unified_matrix_builder import (
    UnifiedMatrixBuilder,
)
from policyengine_us_data.calibration.clone_and_assign import (
    assign_random_geography,
)
from policyengine_us_data.calibration.calibration_utils import (
    create_target_groups,
    drop_target_groups,
    get_geo_level,
    STATE_CODES,
)

db_path = STORAGE_FOLDER / "calibration" / "policy_data.db"
db_uri = f"sqlite:///{db_path}"
dataset_path = STORAGE_FOLDER / "stratified_extended_cps_2024.h5"

In [2]:
sim = Microsimulation(dataset=str(dataset_path))
n_records = sim.calculate("household_id", map_to="household").values.shape[0]

N_CLONES = 3  # keep small for diagnostics
geography = assign_random_geography(n_records, n_clones=N_CLONES, seed=42)

builder = UnifiedMatrixBuilder(
    db_uri=db_uri,
    time_period=2024,
    dataset_path=str(dataset_path),
)

targets_df, X_sparse, target_names = builder.build_matrix(
    geography,
    sim,
    target_filter={"domain_variables": ["aca_ptc", "snap"]},
    hierarchical_domains=["aca_ptc", "snap"],
)

n_total = n_records * N_CLONES
print(f"Records: {n_records:,}, Clones: {N_CLONES}, Total columns: {n_total:,}")
print(f"Matrix shape: {X_sparse.shape}")
print(f"Non-zero entries: {X_sparse.nnz:,}")

Records: 11,999, Clones: 3, Total columns: 35,997
Matrix shape: (1411, 35997)
Non-zero entries: 27,035


## 1.2 Matrix overview

In [3]:
print(f"Targets: {X_sparse.shape[0]}")
print(f"Columns: {X_sparse.shape[1]:,} ({N_CLONES} clones x {n_records:,} records)")
print(f"Non-zeros: {X_sparse.nnz:,}")
print(f"Density: {X_sparse.nnz / (X_sparse.shape[0] * X_sparse.shape[1]):.6f}")

geo_levels = targets_df["geographic_id"].apply(get_geo_level)
level_names = {0: "National", 1: "State", 2: "District"}
for level in [0, 1, 2]:
    n = (geo_levels == level).sum()
    if n > 0:
        print(f"  {level_names[level]}: {n} targets")

Targets: 1411
Columns: 35,997 (3 clones x 11,999 records)
Non-zeros: 27,035
Density: 0.000532
  National: 1 targets
  State: 102 targets
  District: 1308 targets


## 1.3 Anatomy of a row

Each row is one calibration target — a known aggregate (dollar total, household count, person count) that the optimizer tries to match. The row vector's non-zero entries identify which cloned records can contribute to that target.

In [4]:
mid_row = X_sparse.shape[0] // 2
row = targets_df.iloc[mid_row]
print(f"Row {mid_row}: {target_names[mid_row]}")
print(f"  variable: {row['variable']}")
print(f"  geographic_id: {row['geographic_id']}")
print(f"  geo_level: {row['geo_level']}")
print(f"  target value: {row['value']:,.0f}")
print(f"  uprating_factor: {row.get('uprating_factor', 'N/A')}")

Row 705: cd_3402/household_count/[snap>0]
  variable: household_count
  geographic_id: 3402
  geo_level: district
  target value: 48,652
  uprating_factor: 1.0


In [5]:
row_vec = X_sparse[mid_row, :]
nz_cols = row_vec.nonzero()[1]
print(f"Row {mid_row} has {len(nz_cols):,} non-zero columns")

if len(nz_cols) > 0:
    clone_indices = nz_cols // n_records
    record_indices = nz_cols % n_records
    print(f"  Spans {len(np.unique(clone_indices))} clone(s)")
    print(f"  Spans {len(np.unique(record_indices))} unique record(s)")

    first_col = nz_cols[0]
    print(f"\nFirst non-zero column ({first_col}):")
    print(f"  clone_idx: {first_col // n_records}")
    print(f"  record_idx: {first_col % n_records}")
    print(f"  state_fips: {geography.state_fips[first_col]}")
    print(f"  cd_geoid: {geography.cd_geoid[first_col]}")
    print(f"  value: {X_sparse[mid_row, first_col]:.2f}")

Row 705 has 15 non-zero columns
  Spans 3 clone(s)
  Spans 15 unique record(s)

First non-zero column (6161):
  clone_idx: 0
  record_idx: 6161
  state_fips: 34
  cd_geoid: 3402
  value: 1.00


## 1.4 Anatomy of a column

Each column represents one (record, clone) pair. Columns are organized in clone blocks: the first `n_records` columns belong to clone 0, the next to clone 1, and so on. The block formula is:

$$\text{column\_idx} = \text{clone\_idx} \times n_{\text{records}} + \text{record\_idx}$$

In [6]:
col_idx = 1 * n_records + 42  # clone 1, record 42
clone_idx = col_idx // n_records
record_idx = col_idx % n_records
print(f"Column {col_idx}:")
print(f"  clone_idx: {clone_idx}")
print(f"  record_idx: {record_idx}")
print(f"  state_fips: {geography.state_fips[col_idx]}")
print(f"  cd_geoid: {geography.cd_geoid[col_idx]}")
print(f"  block_geoid: {geography.block_geoid[col_idx]}")

col_vec = X_sparse[:, col_idx]
nz_rows = col_vec.nonzero()[0]
print(f"\nThis column has non-zero values in {len(nz_rows)} target rows")
if len(nz_rows) > 0:
    print("First 5 target rows:")
    for r in nz_rows[:5]:
        row = targets_df.iloc[r]
        print(
            f"  row {r}: {row['variable']} "
            f"(geo={row['geographic_id']}, "
            f"val={X_sparse[r, col_idx]:.2f})"
        )

Column 12041:
  clone_idx: 1
  record_idx: 42
  state_fips: 45
  cd_geoid: 4507
  block_geoid: 450410002022009

This column has non-zero values in 0 target rows


## 1.5 Target groups and loss weighting

Target groups partition the rows by (domain, variable, geographic level). Each group contributes equally to the loss function, so hundreds of district-level rows don't drown out a single national row.

In [7]:
target_groups, group_info = create_target_groups(targets_df)

records = []
for gid, info in enumerate(group_info):
    mask = target_groups == gid
    vals = targets_df.loc[mask, "value"]
    records.append(
        {
            "group_id": gid,
            "description": info,
            "n_targets": mask.sum(),
            "min_value": vals.min(),
            "median_value": vals.median(),
            "max_value": vals.max(),
        }
    )

group_df = pd.DataFrame(records)
print(group_df.to_string(index=False))


=== Creating Target Groups ===

National targets:
  Group 0: ACA PTC Person Count = 19,743,689

State targets:
  Group 1: SNAP Household Count (51 targets)
  Group 2: Snap (51 targets)

District targets:
  Group 3: Aca Ptc (436 targets)
  Group 4: ACA PTC Tax Unit Count (436 targets)
  Group 5: SNAP Household Count (436 targets)

Total groups created: 6
 group_id                                                         description  n_targets    min_value  median_value    max_value
        0 Group 0: National ACA PTC Person Count (1 target, value=19,743,689)          1 1.974369e+07  1.974369e+07 1.974369e+07
        1                    Group 1: State SNAP Household Count (51 targets)         51 1.369100e+04  2.772372e+05 3.128640e+06
        2                                    Group 2: State Snap (51 targets)         51 5.670186e+07  1.293585e+09 1.237718e+10
        3                             Group 3: District Aca Ptc (436 targets)        436 5.420354e+06  2.937431e+07 3.880971e+0

## 1.6 Tracing a household across clones

One CPS record appears once per clone (N_CLONES column positions). Each clone places it in a different census block/CD/state, so it contributes to different geographic targets depending on the clone.

In [8]:
snap_values = sim.calculate("snap", map_to="household").values
hh_ids = sim.calculate("household_id", map_to="household").values
example_hh_idx = int(np.where(snap_values > 0)[0][0])
print(f"Example SNAP-receiving household: record index {example_hh_idx}")
print(f"SNAP value: ${snap_values[example_hh_idx]:,.0f}")

clone_cols = [c * n_records + example_hh_idx for c in range(N_CLONES)]
print(f"\nColumn positions across {N_CLONES} clones:")
for col in clone_cols:
    state = geography.state_fips[col]
    cd = geography.cd_geoid[col]
    col_vec = X_sparse[:, col]
    nnz = col_vec.nnz
    abbr = STATE_CODES.get(state, "??")
    print(f"  col {col}: {abbr} (state={state}, CD={cd}) — {nnz} non-zero rows")

Example SNAP-receiving household: record index 23
SNAP value: $70

Column positions across 3 clones:
  col 23: TX (state=48, CD=4829) — 3 non-zero rows
  col 12022: IL (state=17, CD=1707) — 3 non-zero rows
  col 24021: CA (state=6, CD=611) — 3 non-zero rows


## 1.7 Sparsity analysis

In [9]:
total_cells = X_sparse.shape[0] * X_sparse.shape[1]
density = X_sparse.nnz / total_cells
print(f"Total cells: {total_cells:,}")
print(f"Non-zero entries: {X_sparse.nnz:,}")
print(f"Density: {density:.6f}")
print(f"Sparsity: {1 - density:.4%}")

nnz_per_row = np.diff(X_sparse.indptr)
print(f"\nNon-zeros per row:")
print(f"  min:    {nnz_per_row.min():,}")
print(f"  median: {int(np.median(nnz_per_row)):,}")
print(f"  mean:   {nnz_per_row.mean():,.0f}")
print(f"  max:    {nnz_per_row.max():,}")

geo_levels = targets_df["geographic_id"].apply(get_geo_level)
level_names = {0: "National", 1: "State", 2: "District"}
print("\nBy geographic level:")
for level in [0, 1, 2]:
    mask = (geo_levels == level).values
    if mask.any():
        vals = nnz_per_row[mask]
        print(
            f"  {level_names[level]:10s}: "
            f"n={mask.sum():>4d}, "
            f"median nnz={int(np.median(vals)):>7,}, "
            f"range=[{vals.min():,}, {vals.max():,}]"
        )

Total cells: 50,791,767
Non-zero entries: 27,035
Density: 0.000532
Sparsity: 99.9468%

Non-zeros per row:
  min:    0
  median: 10
  mean:   19
  max:    4,241

By geographic level:
  National  : n=   1, median nnz=  4,241, range=[4,241, 4,241]
  State     : n= 102, median nnz=     68, range=[5, 502]
  District  : n=1308, median nnz=     10, range=[0, 21]


## 1.8 Dropping target groups and achievable targets

Some target groups are redundant after hierarchical uprating. For example, state-level SNAP Household Count may be redundant with district-level SNAP Household Count once the districts were reconciled to sum to the state totals.

A target is achievable if at least one household can contribute to it (row sum > 0). Rows with sum = 0 are impossible constraints that the optimizer cannot satisfy.

In [10]:
GROUPS_TO_DROP = [
    ("SNAP Household Count", "State"),
]

targets_filtered, X_filtered = drop_target_groups(
    targets_df, X_sparse, target_groups, group_info, GROUPS_TO_DROP
)

row_sums = np.array(X_filtered.sum(axis=1)).flatten()
achievable_mask = row_sums > 0
n_achievable = achievable_mask.sum()
n_impossible = (~achievable_mask).sum()

print(f"\nAchievable targets: {n_achievable}")
print(f"Impossible targets: {n_impossible}")

X_final = X_filtered[achievable_mask, :]
print(f"\nFinal matrix shape: {X_final.shape}")
print(f"Final non-zero entries: {X_final.nnz:,}")
print(f"This is what the optimizer receives.")

Matrix before: 1411 rows
  DROPPING Group 1: State SNAP Household Count (51 targets) (51 rows)

  KEEPING  Group 0: National ACA PTC Person Count (1 target, value=19,743,689) (1 rows)
  KEEPING  Group 2: State Snap (51 targets) (51 rows)
  KEEPING  Group 3: District Aca Ptc (436 targets) (436 rows)
  KEEPING  Group 4: District ACA PTC Tax Unit Count (436 targets) (436 rows)
  KEEPING  Group 5: District SNAP Household Count (436 targets) (436 rows)

Matrix after: 1360 rows

Achievable targets: 1339
Impossible targets: 21

Final matrix shape: (1339, 35997)
Final non-zero entries: 22,186
This is what the optimizer receives.


### Matrix summary

The calibration matrix pipeline has five steps:

1. **Clone + assign** — `assign_random_geography()` creates N clones of each CPS record, each with a random census block (and derived CD/state).
2. **Build** — `UnifiedMatrixBuilder.build_matrix()` queries targets, applies hierarchical uprating, simulates each clone with its assigned geography, and assembles the sparse CSR matrix.
3. **Groups** — `create_target_groups()` partitions rows for balanced loss weighting.
4. **Sparsity** — Most of the matrix is zero. District-level targets confine non-zeros to clones assigned to that district; national targets span all clones.
5. **Filter** — Remove impossible targets (row sum = 0) before handing to the optimizer.

---
# Part 2: Hierarchical Uprating

Calibration targets in `policy_data.db` come from different sources, at different geographic levels, and from different time periods. Before we can use them, two adjustments are needed:

1. **Uprating factor (UF)**: Bridges the time gap between the source data's period and the calibration year. For most domains, dollar-valued targets use CPI and count targets use population growth. For **ACA PTC**, we use real state-level enrollment and average APTC changes from CMS/KFF data, giving each state its own UF.

2. **Hierarchy inconsistency factor (HIF)**: Corrects for the fact that district-level totals from one source may not sum to the state-level total from another. This is a pure base-year geometry correction with no time dimension.

These two factors are **separable by linearity**. For each congressional district row:

$$\text{value} = \text{original\_value} \times \text{HIF} \times \text{UF}$$

where $\text{HIF} = S_{\text{base}} \;/\; \sum_i CD_{i,\text{base}}$ and the sum constraint holds:

$$\sum_i (CD_i \times \text{HIF} \times \text{UF}) = \text{UF} \times S_{\text{base}} = S_{\text{uprated}}$$

Two example domains:
- **ACA PTC** (IRS data): Districts sum exactly to state totals, so HIF = 1.0 everywhere. The UF varies by state, reflecting real enrollment and APTC changes between 2022 and 2024.
- **SNAP** (USDA data): District household counts substantially undercount the state administrative totals, so HIF > 1 (often 1.2 to 1.7). The SNAP data is already at the target period, so UF = 1.0.

## 2.1 Raw targets and generic uprating

In [11]:
DOMAINS = ["aca_ptc", "snap"]

raw = builder._query_targets({"domain_variables": DOMAINS})

summary = (
    raw.groupby(["domain_variable", "geo_level", "variable", "period"])
    .agg(count=("value", "size"), total_value=("value", "sum"))
    .reset_index()
)
summary

,domain_variable,geo_level,variable,period,count,total_value
0,aca_ptc,district,aca_ptc,2022,436,1.419185e+10
1,aca_ptc,district,tax_unit_count,2022,436,6.848330e+06
2,aca_ptc,national,aca_ptc,2022,1,1.419185e+10
3,aca_ptc,national,person_count,2024,1,1.974369e+07
4,aca_ptc,national,tax_unit_count,2022,1,6.848330e+06
5,aca_ptc,state,aca_ptc,2022,51,1.419185e+10
6,aca_ptc,state,tax_unit_count,2022,51,6.848330e+06
7,snap,district,household_count,2024,436,1.563268e+07
8,snap,state,household_count,2024,51,2.217709e+07
9,snap,state,snap,2024,51,9.365787e+10


In [12]:
params = sim.tax_benefit_system.parameters
uprating_factors = builder._calculate_uprating_factors(params)

for (yr, kind), f in sorted(uprating_factors.items()):
    if f != 1.0:
        print(f"  {yr} -> 2024 ({kind}): {f:.6f}")

  2022 -> 2024 (cpi): 1.101889
  2022 -> 2024 (pop): 1.020415
  2023 -> 2024 (cpi): 1.035512
  2023 -> 2024 (pop): 1.010947
  2025 -> 2024 (cpi): 0.970879
  2025 -> 2024 (pop): 0.990801


## 2.2 Hierarchical reconciliation

For each (state, variable) pair within a domain:

- **HIF** = `state_original / sum(cd_originals)` — pure base-year correction
- **UF** = state-specific uprating factor:
  - For **ACA PTC**: loaded from `aca_ptc_multipliers_2022_2024.csv` (CMS/KFF enrollment data)
  - For other domains: national CPI/pop factors as fallback

In [13]:
raw["original_value"] = raw["value"].copy()
raw["uprating_factor"] = raw.apply(
    lambda r: builder._get_uprating_info(
        r["variable"], r["period"], uprating_factors
    )[0],
    axis=1,
)
raw["value"] = raw["original_value"] * raw["uprating_factor"]

result = builder._apply_hierarchical_uprating(raw, DOMAINS, uprating_factors)

In [14]:
sample_states = {6: "CA", 48: "TX", 36: "NY"}


def show_reconciliation(result, raw, domain, sample_states):
    domain_rows = result[result["domain_variable"] == domain]
    cd_domain = domain_rows[domain_rows["geo_level"] == "district"]
    if cd_domain.empty:
        print("  (no district rows)")
        return
    for fips, abbr in sample_states.items():
        cd_state = cd_domain[
            cd_domain["geographic_id"].apply(
                lambda g, s=fips: int(g) // 100 == s
                if g not in ("US",)
                else False
            )
        ]
        if cd_state.empty:
            continue
        for var in sorted(cd_state["variable"].unique()):
            var_rows = cd_state[cd_state["variable"] == var]
            hif = var_rows["hif"].iloc[0]
            uf = var_rows["state_uprating_factor"].iloc[0]
            cd_sum = var_rows["value"].sum()
            print(
                f"  {abbr} {var:20s}  "
                f"hif={hif:.6f}  "
                f"uprating={uf:.6f}  "
                f"sum(CDs)={cd_sum:>14,.0f}"
            )


print("ACA PTC (HIF=1.0, state-varying UF):")
show_reconciliation(result, raw, "aca_ptc", sample_states)
print("\nSNAP (HIF>1, UF=1.0):")
show_reconciliation(result, raw, "snap", sample_states)

ACA PTC (HIF=1.0, state-varying UF):
  CA aca_ptc               hif=1.000000  uprating=1.209499  sum(CDs)= 3,332,007,010
  CA tax_unit_count        hif=1.000000  uprating=1.055438  sum(CDs)=     1,302,653
  TX aca_ptc               hif=1.000000  uprating=1.957664  sum(CDs)= 2,270,594,110
  TX tax_unit_count        hif=1.000000  uprating=1.968621  sum(CDs)=     1,125,834
  NY aca_ptc               hif=1.000000  uprating=1.343861  sum(CDs)= 2,049,797,288
  NY tax_unit_count        hif=1.000000  uprating=1.075089  sum(CDs)=       593,653

SNAP (HIF>1, UF=1.0):
  CA household_count       hif=1.681273  uprating=1.000000  sum(CDs)=     3,128,640
  TX household_count       hif=1.244524  uprating=1.000000  sum(CDs)=     1,466,107
  NY household_count       hif=1.344447  uprating=1.000000  sum(CDs)=     1,707,770


## 2.3 Verification: sum(CDs) == uprated state

The core invariant: for every (state, variable) pair that has district rows, the sum of reconciled district values must equal the uprated state total.

In [15]:
all_ok = True
checks = 0
for domain in DOMAINS:
    domain_result = result[result["domain_variable"] == domain]
    cd_result = domain_result[domain_result["geo_level"] == "district"]
    if cd_result.empty:
        continue
    for fips, abbr in sorted(STATE_CODES.items()):
        cd_rows = cd_result[
            cd_result["geographic_id"].apply(
                lambda g, s=fips: int(g) // 100 == s
                if g not in ("US",)
                else False
            )
        ]
        if cd_rows.empty:
            continue
        for var in cd_rows["variable"].unique():
            var_rows = cd_rows[cd_rows["variable"] == var]
            cd_sum = var_rows["value"].sum()
            st = raw[
                (raw["geo_level"] == "state")
                & (raw["geographic_id"] == str(fips))
                & (raw["variable"] == var)
                & (raw["domain_variable"] == domain)
            ]
            if st.empty:
                continue
            state_original = st["original_value"].iloc[0]
            state_uf = var_rows["state_uprating_factor"].iloc[0]
            expected = state_original * state_uf
            ok = np.isclose(cd_sum, expected, rtol=1e-6)
            checks += 1
            if not ok:
                print(
                    f"  FAIL [{domain}] {abbr} {var}: "
                    f"sum(CDs)={cd_sum:.2f} != "
                    f"expected={expected:.2f}"
                )
                all_ok = False

print(
    f"  {checks} checks across {len(DOMAINS)} domains: "
    + ("ALL PASSED" if all_ok else "SOME FAILED")
)

  153 checks across 2 domains: ALL PASSED


---
# Part 3: H5 Builder Reference

`build_h5` is the single function that produces all local-area H5 datasets (national, state, district, city). It lives in `policyengine_us_data/calibration/publish_local_area.py`.

## 3.1 Signature

```python
def build_h5(
    weights: np.ndarray,
    geography: GeographyAssignment,
    dataset_path: Path,
    output_path: Path,
    cd_subset: List[str] = None,
    county_filter: set = None,
    takeup_filter: List[str] = None,
) -> Path:
```

## 3.2 Parameter Semantics

| Parameter | Type | Purpose |
|---|---|---|
| `weights` | `np.ndarray` | Clone-level weight vector, shape `(n_clones * n_hh,)` |
| `geography` | `GeographyAssignment` | Geography assignment from `assign_random_geography` |
| `dataset_path` | `Path` | Path to base dataset H5 file |
| `output_path` | `Path` | Where to write the output H5 file |
| `cd_subset` | `List[str]` | If provided, only include clones for these CDs |
| `county_filter` | `set` | If provided, scale weights by P(target counties \| CD) for city datasets |
| `takeup_filter` | `List[str]` | List of takeup variables to re-randomize |

## 3.3 How `cd_subset` Controls Output Level

- **National** (`cd_subset=None`): All CDs included — produces a full national dataset.
- **State** (`cd_subset=[CDs in state]`): Filter to CDs whose FIPS prefix matches the state.
- **District** (`cd_subset=[single_cd]`): Single CD — produces a district dataset.
- **City** (`cd_subset=[NYC CDs]` + `county_filter=NYC_COUNTIES`): Multiple CDs with county filtering. The `county_filter` scales weights by the probability that a household in each CD falls within the target counties.

## 3.4 Internal Pipeline

1. **Load base simulation** — One `Microsimulation` loaded from `dataset_path`. Entity arrays and membership mappings extracted.
2. **Reshape weights** — The flat weight vector is reshaped to `(n_clones, n_hh)`.
3. **CD subset filtering** — Clones for CDs not in `cd_subset` are zeroed out.
4. **County filtering** — If `county_filter` is set, each clone's weight is scaled by `P(target_counties | CD)` via `get_county_filter_probability()`.
5. **Identify active clones** — `np.where(W > 0)` finds all nonzero entries. Each represents a distinct household clone.
6. **Clone entity arrays** — Entity arrays (household, person, tax_unit, spm_unit, family, marital_unit) are cloned using fancy indexing on the base simulation arrays.
7. **Reindex entity IDs** — All entity IDs are reassigned to be globally unique. Cross-reference arrays (e.g., `person_household_id`) are updated accordingly.
8. **Derive geography** — Block GEOIDs are mapped to state FIPS, county, tract, CBSA, etc. via `derive_geography_from_blocks()`. Unique blocks are deduplicated for efficiency.
9. **Recalculate SPM thresholds** — SPM thresholds are recomputed using `calculate_spm_thresholds_vectorized()` with the clone's CD-level geographic adjustment factor.
10. **Rerandomize takeup** (optional) — If enabled, takeup booleans are redrawn per census block using `apply_block_takeup_to_arrays()`.
11. **Write H5** — All variable arrays are written to the output file.

## 3.5 Usage Examples

### National
```python
build_h5(
    weights=w,
    geography=geography,
    dataset_path=Path("base.h5"),
    output_path=Path("national/US.h5"),
)
```

### State
```python
state_cds = [cd for cd in geography.cd_geoid if int(cd) // 100 == 6]
build_h5(
    weights=w,
    geography=geography,
    dataset_path=Path("base.h5"),
    output_path=Path("states/CA.h5"),
    cd_subset=list(set(state_cds)),
)
```

### District
```python
build_h5(
    weights=w,
    geography=geography,
    dataset_path=Path("base.h5"),
    output_path=Path("districts/CA-12.h5"),
    cd_subset=["0612"],
)
```

### City (NYC)
```python
from policyengine_us_data.calibration.publish_local_area import (
    NYC_COUNTIES, NYC_CDS,
)

build_h5(
    weights=w,
    geography=geography,
    dataset_path=Path("base.h5"),
    output_path=Path("cities/NYC.h5"),
    cd_subset=NYC_CDS,
    county_filter=NYC_COUNTIES,
)
```